In [ ]:
# Import libraries
import sys
import os

import numpy as np
import h5py

import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim 
from torch.utils.data import DataLoader, TensorDataset, random_split

from pathlib import Path

from tqdm import tqdm

In [ ]:
#---- INITIALISE PATHS ----
class pathsBib:
    data_path = 'data/'
    model_path = 'model/'
    res_path = 'res/'


def init_path():
    """
    Returns:
        is_init_path()  :   (bool) if initialisation is successful

    """

    is_init_path = False
    try:
        print("#"*30)
        print(f"Start initialization of paths")
        path_list = [i for _,i in pathsBib.__dict__.items() if type(i)==str and "/" in i]
        print(path_list)
        for pth in path_list:
            Path(pth).mkdir(exist_ok=True)
            print(f"INIT:\t{pth}\tDONE")
        print("#"*30)
        is_init_path = True
    except:
        print(f"ERROR: failed to initialise path. Please, check setup for your path!")
        sys.exit()



#---- MAKE DATALOADER ----
class dataclass:
    def __init__(self, input_len, output_len, batch_size, train_split, scaling):
        """
        Args:
            input_len       :   (int) length of input sequence
            output_len      :   (int) length of output sequence
            batch_size      :   (int) batch size
            train_split     :   (float) ratio of train and validation split (if 1, no validation)
            scaling         :   (str) type of scaling (minmax, standard)

        """
        self.input_len = input_len
        self.output_len = output_len
        self.batch_size = batch_size
        self.train_split = train_split
        self.scaling = scaling


    def get_data(self):
        """
        Create Dataloader for training and validation (if applicable)

        """
        try:
            f = h5py.File(pathsBib.data_path + 'POD_space.h5py', 'r')
            data = np.array(f['train'])
            f.close()
            if self.scaling:
                data_norm = dataclass.normalise(data, self.scaling)     
        except:
            print(f"ERROR: failed to find data. Please, check path or file!")
            sys.exit()
        
        X, Y = dataclass.make_Sequence(self, data=data_norm)
        self.train_dl, self.val_dl = dataclass.make_Dataloader(torch.from_numpy(X), torch.from_numpy(Y),
                                                    batch_size=self.batch_size,
                                                    drop_last=False,
                                                    train_split=self.train_split)
        print(f"INFO: DataLoader has been generated!")
        del data, data_norm, X, Y
        return self.train_dl, self.val_dl


    def make_Sequence(self, data):
        """
        Generate sliding window data

        Returns:
            X   :   (arr) Input data
            Y   :   (arr) Output data

        """
        if len(data.shape) <=2:
            data = np.expand_dims(data,0)
        nSamples = data.shape[-1] - self.input_len - self.output_len + 1
        # Initialise return arrays
        X = np.empty([nSamples, self.input_len, data.shape[1]])
        Y = np.empty([nSamples, self.output_len, data.shape[1]]) 
        k = 0
        for i in tqdm(np.arange(data.shape[0])):
            for j in np.arange(data.shape[-1] - self.input_len - self.output_len):
                X[k] = np.transpose(data[i, :, j       :j+self.input_len]) # put sequence first for LSTM
                Y[k] = np.transpose(data[i, :, j+self.input_len:j+self.input_len+self.output_len]) 
                k    = k + 1

        print(f"The training data has been generated with shape of {X.shape, Y.shape}")

        return X, Y


    def make_Dataloader(X, y, batch_size, drop_last=False, train_split=1):
        """
        Args:
            drop_last           :   (bool) if True, drop the last batch if it does not have same number of samples

        Return:
            train_dl, val_dl    :   train and validation DataLoader

        """
        dataset = TensorDataset(X, y)

        len_d = len(dataset)
        train_size = int(train_split * len_d)
        valid_size = len_d - train_size

        train_d, val_d = random_split(dataset, [train_size, valid_size])

        train_dl = DataLoader(train_d, batch_size=batch_size, drop_last=drop_last, shuffle=True)
        if valid_size > 0:
            val_dl = DataLoader(val_d, batch_size=batch_size, drop_last=drop_last, shuffle=True)
        else:
            val_dl = None

        return train_dl, val_dl


    def normalise(data, scaling):
        """
        Returns:
            data_norm   :   (arr) normalised data

        """
        if scaling == "minmax":
            minval = np.min(data)
            maxval = np.max(data)
            np.save(pathsBib.data_path + f'minmax-scaling.npy', [minval, maxval])
            data_norm = (data - minval) / (maxval - minval)
        elif scaling == "standard":
            meanval = np.mean(data)
            stdval = np.std(data)
            np.save(pathsBib.data_path + f'standard-scaling.npy', [meanval, stdval])
            data_norm = (data - meanval) / stdval
        else:
            print(f"ERROR: failed to normalise data. Please, check scaling type!")
            sys.exit()

        return data_norm


    def reverse_normalise(data, scaling):
        """
        Returns:
            data    :   (arr) non-normalised data

        """
        if scaling == "minmax":
            minval, maxval = np.load(pathsBib.data_path + f'minmax-scaling.npy')
            data = data * (maxval - minval) + minval
        elif scaling == "standard":
            meanval, stdval = np.load(pathsBib.data_path + f'standard-scaling.npy')
            data = data * stdval + meanval
        else:
            print(f"ERROR: failed to reverse normalise data. Please, check scaling type!")
            sys.exit()

        return data



# ---- NETWORKS ----
class LSTM(nn.Module):
    def __init__(self, input_dim, lstm_dim, num_layer=1, dropout=0.0):
        """ 
        Long-Short Term Memory (LSTM) network

        Args:
            input_dim   :   (int) input dimension of model 
            lstm_dim    :   (int) hidden dimension of LSTM
            num_layer   :   (int) number of LSTM layers
            dropout     :   (float) dropout rate between LSTM layers (if num_layer > 1)

        """
        super(LSTM, self).__init__()
        self.lstm_dim = lstm_dim
        self.num_layer = num_layer

        self.lstm = nn.LSTM(input_size=input_dim, hidden_size=self.lstm_dim, num_layers=self.num_layer, dropout=dropout, batch_first=True)


    def init_hidden(self, batch_size, device):
        hidden = torch.zeros(self.num_layer,
                            batch_size,
                            self.lstm_dim).to(device)

        cell  =  torch.zeros(self.num_layer,
                            batch_size,
                            self.lstm_dim).to(device)
        return hidden, cell
        

    def forward(self, input_tensor):
        """
        Returns:
            output      :   (tensor) output from last layer (shape [batch_size, seq_len, hidden_dim])
         
        """
        hidden, cell = self.init_hidden(input_tensor.shape[0], device=input_tensor.device)

        output, (_, _) = self.lstm(input_tensor, (hidden.detach(), cell.detach())) 

        return output


class ResLSTMBlock(nn.Module):
    def __init__(self, lstm_dim, mlp_dim):
        super(ResLSTMBlock, self).__init__()

        self.LSTM = LSTM(input_dim=lstm_dim, lstm_dim=lstm_dim, num_layer=1, dropout=0.0)

        self.mlp = nn.Sequential(
            nn.Linear(lstm_dim, mlp_dim),
            nn.ReLU(),
            nn.Linear(mlp_dim, lstm_dim),
        )

        self.layernorm = nn.LayerNorm(lstm_dim)


    def forward(self, input_tensor):
        input_tensor_ln = self.layernorm(input_tensor)
        out_lstm = self.LSTM(input_tensor_ln)

        out_ffn = self.mlp(out_lstm)

        output = out_ffn + input_tensor # skip connection; pre-LN: output = f(LN(input)) + input
        input_tensor = output # input for next block
        return input_tensor
    

class ResNet(nn.Module):
    def __init__(self, input_dim, lstm_dim, mlp_dim, num_levels):
        """ 
        Long-Short Term Memory with residual (ResLSTM) network 

        Args:
            input_dim   :   (int) input dimension of model
            lstm_dim    :   (int) hidden dimension of LSTM
            mlp_dim     :   (int) hidden dimension of MLP
            num_levels  :   (int) number of ResLSTM blocks

        """
        super(ResNet, self).__init__()

        self.proj = nn.Linear(input_dim, lstm_dim) # projection layer

        print(f"INFO: the model has {num_levels} ResLSTM blocks")
        layers = []
        for i in range(num_levels):
            layers += [ResLSTMBlock(lstm_dim, mlp_dim)]
        self.ResNet = nn.Sequential(*layers)

    
    def forward(self, input_tensor):
        input_tensor_proj = self.proj(input_tensor)
        output = self.ResNet(input_tensor_proj)
        return output


class NN(nn.Module):
    def __init__(self, model_params):
        super(NN, self).__init__()

        self.output_len = model_params['output_len']

        self.network = ResNet(input_dim=model_params['input_dim'],
                              lstm_dim=model_params['lstm_dim'],
                              mlp_dim=model_params['mlp_dim'],
                              num_levels=model_params['num_layer'])
        
        self.out1 = nn.Linear(model_params['lstm_dim'], model_params['lstm_dim'] * model_params['output_len'])
        self.out2 = nn.Linear(model_params['lstm_dim'], model_params['output_dim'])

    
    def forward(self, input_tensor):
        out_net = self.network(input_tensor)

        in_out = out_net[:, -1, :].unsqueeze(1) # time pooling (get last sample in the sequence)
        out = self.out1(in_out)

        out_reshape = out.view(out.shape[0], self.output_len, out_net.shape[2]) # reshape to [batch_size, output_len, lstm_dim]
        output = self.out2(out_reshape) 
        return output



# ---- TRAINING LOOP ----
def train(device, model, train_dl, loss_fn, optimizer, scheduler=None, num_epoch=100, eps_huber=0.0, val_dl=None):
    """
    Args: 
        device      :   the device for training (it) should match the model's device!)
        model       :   the model to be trained
        train_dl    :   dataloader for training
        loss_fn     :   loss function     
        optimizer   :   optimizer function
        scheduler   :   scheduler function
        num_epoch   :   (int) number of epochs
        eps_huber   :   (float) threshold for Huber loss (if 0, use MSE loss)
        val_dl      :   Dataloader for validation (if applicable)

    Returns:
        history     :   (dict) contains training and validation (if applicable) losses

    """
    history = {}
    history["train_loss"] = []

    if val_dl:
        history["val_loss"] = []

    model.to(device)

    for epoch in range(num_epoch):
        model.train() # change to training mode

        loss_val = 0; num_batch = 0
        for batch in tqdm(train_dl):
            x, y = batch
            x = x.to(device).float(); y = y.to(device).float()
            optimizer.zero_grad()

            pred = model(x)

            if eps_huber != 0: 
                if num_batch != 0 and num_batch % (len(train_dl)-1) == 0 and epoch % 10 == 0:
                    with torch.no_grad():
                        e = (pred - y).abs()
                        frac_mae = (e > eps_huber).float().mean().item()
                    print(f"INFO: fraction of samples in MAE regime: {frac_mae:.3f}")

            loss = loss_fn(pred, y)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0) # gradient clipping

            optimizer.step()

            loss_val += loss.item()/x.shape[0]
            num_batch += 1
        
        if scheduler is not None:
            scheduler.step()

        history["train_loss"].append(loss_val/num_batch)
    
       
        if val_dl:
            model.eval() # change to evaluation mode

            loss_val = 0; num_batch = 0
            for batch in (val_dl):
                x, y = batch
                x = x.to(device).float(); y = y.to(device).float()

                pred = model(x) 
                loss = loss_fn(pred, y) 

                loss_val += loss.item()/x.shape[0]
                num_batch += 1

            history["val_loss"].append(loss_val/num_batch)

        train_loss = history["train_loss"][-1]
        if val_dl:
            val_loss = history["val_loss"][-1]
            print(f"At Epoch    = {epoch+1},\n"
                  f"Train_loss  = {train_loss},\n"
                  f"Val_loss    = {val_loss}" 
            )
            if scheduler is not None:
                print(f"LR          = {scheduler.get_last_lr()[0]:.4e}")
        else:
            print(f"At Epoch    = {epoch+1},\n"
                  f"Train_loss  = {train_loss}" 
            )
            if scheduler is not None:
                print(f"LR          = {scheduler.get_last_lr()[0]:.4e}")

    return history



# ---- TEST ----
class testclass:
    def __init__(self, device, input_len, output_len, n_test, n_modes, scaling):
        self.device = device
        self.input_len = input_len
        self.output_len = output_len
        self.n_test = n_test
        self.n_modes = n_modes
        self.scaling = scaling
        

    def load_pretrain_model(self, model_params):
        """
        Load state dict and history of pre-trained model
    
        Returns:
            stat_dict   :   learnable parameters 
            history     :   history of training

        """
        model_path = pathsBib.model_path + f"POD-rDL_statedict-history" + ".pt"

        try:
            ckpoint = torch.load(model_path, weights_only=False, map_location=self.device)
        except:
            print("ERROR: model not found!")
            sys.exit()

        stat_dict = ckpoint['model']

        self.model = NN(model_params)
        self.model.load_state_dict(state_dict=stat_dict)
        self.history = ckpoint['history']

        print(f'INFO: the state dict has been loaded!')
        print(self.model.eval)

        return self.model


    def test(self):
        try:
            f = h5py.File(pathsBib.data_path + 'POD_space.h5py', 'r')
            data = np.transpose(np.array(f['train']))
            f.close()
            if self.scaling:
                data_norm = dataclass.normalise(data, self.scaling)
            data_norm = data_norm[-self.input_len:, :] # get last input_len samples to initialise encoder
        except:
            print(f"ERROR: failed to find data. Please, check path or file!")
            sys.exit()

        self.model.eval()
        self.model.to(self.device)

        print(f"INFO: testing model")

        self.output = np.concatenate((data_norm, np.zeros((self.n_test, data_norm.shape[1]))), axis=0) # initialize prediction array

        print(f"INFO: starting autoregressive rollout for {self.n_test} steps")
        for i in tqdm(range(self.input_len, self.output.shape[0], self.output_len)):
            feature = self.output[None, i-self.input_len:i, :]

            x = torch.from_numpy(feature)
            x = x.float().to(self.device)
            pred = self.model(x)

            pred = pred.cpu().detach().numpy()

            self.output[i:i+self.output_len,:] = pred[0,:,:]

        if self.scaling:
            self.output = dataclass.reverse_normalise(self.output, self.scaling)

        self.output = self.output[self.input_len:,:] # remove training data from prediction array

        # Save results
        np.savez_compressed(
        file = pathsBib.res_path + 'POD-rDL_Preds.npz',
        out = self.output
        )


    def plot_loss(self):
        """
        Plot training and validation (if applicable) losses

        """
        fig, axs = plt.subplots(1, 1, figsize=(10, 4))

        axs.plot(self.history["train_loss"], color='blue', linestyle='-', linewidth=1.5, marker='o', markersize=5)
        if len(self.history["val_loss"]) != 0:
            axs.plot(self.history["val_loss"], color='red', linestyle='-', linewidth=1.5, marker='^', markersize=5)
        axs.set_yscale('log')
        axs.grid(True, which='both')
        axs.set_xlim([-1, len(self.history["train_loss"])])
        axs.tick_params(axis='both', which='major', labelsize=14)
        axs.set_xlabel("Training epoch", fontsize=14)
        axs.set_ylabel("Loss", fontsize=14)

        axs.legend(["Train", "Validation"], fontsize=14)


    def plot_pred(self):
        """
        Plot prediction

        """
        import sys

        try:
            f = h5py.File(pathsBib.data_path + 'POD_space.h5py', 'r')
            data = np.transpose(np.array(f['test']))
        except:
            print(f"ERROR: failed to find data. Please, check path or file!")
            sys.exit()
      
        fig, axs = plt.subplots(5, 1, figsize=(10, 4), sharex=True)
        for i, ax in enumerate(axs):
            ax.plot(data[:, i], color='black', linestyle='-', linewidth=1.5)
            ax.plot(self.output[:, i], color='blue', linestyle='-', linewidth=1.5)
            ax.tick_params(axis='both', which='major', labelsize=10)
            ax.set_ylabel(rf"$POD{i+1}$", fontsize=10)

        axs[-1].set_xlabel(rf"Prediction step",fontsize=10)


In [ ]:
# ---- CONFIGURATION ----
class config:
    n_modes = 25 # number of POD modes 
    n_test = 80

    input_len = 64
    batch_size = 16
    train_split = 0.8
    scaling = "standard" # "minmax", "standard"

    model_params = {
    'input_dim' :   n_modes,
    'lstm_dim'  :   64,    
    'mlp_dim'   :   256, 
    'num_layer' :   3, # number of ResLSTM blocks
    'output_len':   1,
    'output_dim':   n_modes
    }

    eps_huber = 0.0 # threshold for Huber loss; if 0, use MSE loss
    lr = 1e-3
    schlr = True # if True, use learning rate scheduler (exponential decay)
    num_epoch = 1000

In [ ]:
# Create environment
init_path()

In [ ]:
DL = dataclass(config.input_len, config.model_params["output_len"], config.batch_size, config.train_split, config.scaling)
train_dl, val_dl = DL.get_data()
print(f"INFO: number of training batches: {len(train_dl)}")
if val_dl:
    print(f"INFO: number of validation batches: {len(val_dl)}")

In [ ]:
model = NN(config.model_params)
print(model)
NumPara = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"INFO: the model has been generated, the number of parameter is {NumPara}")

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"INFO: the device has been assigned to {device} ")

In [ ]:
loss_fn = nn.HuberLoss(delta=config.eps_huber) if config.eps_huber != 0 else nn.MSELoss()

optimizer = optim.Adam(model.parameters(), lr=config.lr)
if config.schlr == True:
    scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.99)
else:
    scheduler = None

In [ ]:
# Check if pre-trained model exists (avoid training)
model_path = pathsBib.model_path + f"POD-rDL_statedict-history" + ".pt"
if not os.path.isfile(model_path):
    print(f"INFO: start training!")
    history = train(device, model, train_dl, loss_fn, optimizer, scheduler, num_epoch=config.num_epoch, eps_huber=config.eps_huber, val_dl=val_dl)
    print(f"INFO: training finished!")

    check_point = {"model":model.state_dict(),
                   "history":history,
                   }
    
    torch.save(check_point, model_path)
    print(f"INFO: the checkpoint has been saved!")

In [ ]:
print(f"INFO: start testing!")
TT = testclass(device, config.input_len, config.model_params['output_len'], config.n_test , config.n_modes, config.scaling)

model = TT.load_pretrain_model(config.model_params)
print(f"INFO: the model has been loaded, the number of parameter is {NumPara}")

TT.test()

In [ ]:
TT.plot_loss()

In [ ]:
TT.plot_pred()